# HLS Video Player — Google Colab 起動ノート

このノートブックは hls-video-player を Colab 上で動作させます。**コードベースは Google Drive から取得**、メディア（ソース動画と変換出力）も Drive に永続化します。

## 事前準備（初回のみ）

Google Drive に次のディレクトリレイアウトでコードを配置してください（ローカルの `hls-video-player/` ディレクトリをそのままアップロード）:

```
MyDrive/
└── hls-video-player/
    ├── pyproject.toml
    ├── app/
    ├── hls_video/
    ├── static/
    └── media/
        ├── source/   ← 変換元 MP4 を置く（永続）
        ├── hls/      ← 変換出力（永続、再生成可能）
        └── sprites/  ← スプライト + VTT（永続、再生成可能）
```

コード更新は手元で編集して Drive に上書き同期するだけで済みます。

## 実行手順

1. FFmpeg + Python 依存インストール
2. Drive マウント
3. Google Drive からコードベースをコピー
4. Python パッケージとしてインストール
5. MEDIA_ROOT を Drive に設定
6. アプリ起動

## 1. FFmpeg + Python 依存

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1
# NVENC (h264_nvenc) が利用可能か確認（ランタイム: T4 GPU を選ぶと出ます）
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E 'h264_nvenc|libx264' || true
!nvidia-smi -L 2>/dev/null || echo '(GPU ランタイム未選択: CPU エンコードになります)'
!pip install -q gradio fastapi uvicorn python-multipart

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Google Drive からコードベースをコピー

Drive 上のコードは FUSE 経由で I/O が遅いため、Colab ローカル (`/content/hls-video-player`) に
複製してから使う。ただし `media/` はコピーせず、Drive 上の実体へ **シンボリックリンク** を張る。
→ アプリケーションは Colab 上で動き、動画ファイル類は Drive に永続化される。

再実行時も毎回丸ごと上書きコピーし直す。手元で更新 → Drive に同期 → Colab で本セルを再実行、で最新に揃う。

In [ ]:
import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/hls-video-player'
LOCAL_ROOT = '/content/hls-video-player'

assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} が見つかりません。MyDrive 直下に hls-video-player/ を配置してください。'
)

# コード本体を /content/hls-video-player へ丸ごとコピー (media は除外)
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(
    DRIVE_ROOT,
    LOCAL_ROOT,
    ignore=shutil.ignore_patterns(
        'media',            # 実体は Drive 側、直後にシンボリックリンクを張る
        '.git', '.claude',
        '__pycache__', '.pytest_cache', '.venv', '*.egg-info',
        'node_modules',
    ),
)

# Drive 上の media/{source,hls,sprites} を用意（初回実行時のため）
drive_media = f'{DRIVE_ROOT}/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{drive_media}/{sub}', exist_ok=True)

# /content/hls-video-player/media → /content/drive/MyDrive/hls-video-player/media (symlink)
local_media = f'{LOCAL_ROOT}/media'
if os.path.islink(local_media) or os.path.exists(local_media):
    os.remove(local_media) if os.path.islink(local_media) else shutil.rmtree(local_media)
os.symlink(drive_media, local_media)

%cd {LOCAL_ROOT}
!ls -l | head -20
print(f'\nmedia symlink: {local_media} -> {os.readlink(local_media)}')

## 4. Python パッケージとしてインストール

`pip install -e .` でコピーしたソースを editable モードでインストール。これで `from hls_video import ...` / `from app import ...` が通るようになる。

In [ ]:
!pip install -q -e .

## 5. MEDIA_ROOT を設定

`{LOCAL_ROOT}/media` は Drive 上の実体へのシンボリックリンク (セル 3 で作成)。
アプリケーション側からはローカルパスに見えるが、読み書きは Drive 実体に反映されるため
セッションを跨いで永続化される。

In [ ]:
import os

# symlink 経由で Drive 実体を指すので、アプリからはローカルパスに見える
os.environ['MEDIA_ROOT'] = f'{LOCAL_ROOT}/media'
os.environ['PORT'] = '7860'

# --- 変換パフォーマンス ---
# NVENC が使える環境なら自動選択 (FFMPEG_HWACCEL=auto)。
# 明示したい場合は 'nvenc' / 'cpu' を指定。
os.environ['FFMPEG_HWACCEL'] = 'auto'
os.environ['FFMPEG_NVENC_PRESET'] = 'p4'          # p1 (最速) .. p7 (最高画質)
os.environ['FFMPEG_PRESET'] = 'ultrafast'         # CPU fallback 時の libx264 preset
os.environ['FFMPEG_THREADS'] = '0'                # 0 = ffmpeg auto
# os.environ['FFMPEG_VARIANTS'] = '720p,360p'     # 必要なら 2 本に絞って更に高速化
os.environ['MAX_CONCURRENT_JOBS'] = '1'           # 1 本ずつ順に変換（長尺向け）

print('MEDIA_ROOT    =', os.environ['MEDIA_ROOT'])
print('FFMPEG_HWACCEL =', os.environ['FFMPEG_HWACCEL'])
print('symlink →', os.readlink(os.environ['MEDIA_ROOT']))
!ls {os.environ['MEDIA_ROOT']}

## 6. 起動（外部アクセス可）

FastAPI (Gradio + `/hls` + `/sprites` + `/api` + `/player`) を port 7860 に立て、
Gradio の `setup_tunnel` で `*.gradio.live` の **公開 URL** を発行する。

**なぜ `demo.launch(share=True)` を使わないか:** `launch()` は Gradio 単体用に
別プロセスを立てるので、同じ FastAPI に載せた `/hls/*` / `/sprites/*` の
静的配信が share URL で見えなくなる。`setup_tunnel` で FastAPI 本体 (7860) を
直接トンネル化すれば全ルートが単一の公開 URL 配下でアクセスできる。

In [ ]:
import gradio as gr
from fastapi import FastAPI

from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

# FastAPI 本体をバックグラウンドで port 7860 に立てる
# 再起動セル（末尾）から停止できるよう Server オブジェクトはグローバルに保持する
import threading, time, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)  # uvicorn の起動待ち

# Gradio の共有トンネル (*.gradio.live) を port 7860 に張る
import secrets
from gradio.networking import setup_tunnel

share_url = setup_tunnel(
    local_host='localhost',
    local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None,
    share_server_tls_certificate=None,
)

print(f'\n🌐 公開 URL: {share_url}')
print('   /          → Gradio UI')
print('   /hls/*     → HLS プレイリスト・セグメント')
print('   /sprites/* → スプライト・WebVTT')
print('   /api, /player, /static も同じ URL 配下で利用可能')
print('\n💻 Colab 内からのアクセス: http://localhost:7860')

### 公開 URL のライフサイクル

- `*.gradio.live` のトンネルは **72 時間** で自動失効（Gradio 側の仕様）
- Colab ランタイム切断時にも失効する
- 再発行は上のセルを再実行するだけ（新しい URL が振られる）
- 長く稼働させ続けたい場合は Colab Pro + `Keep session active` や ngrok などの外部トンネルを検討

### トラブルシューティング

| 症状 | 対処 |
|---|---|
| 公開 URL を開くと 502 / 接続エラー | uvicorn が落ちた可能性。`!ss -tlnp \| grep 7860` で確認し、セル 6 を再実行 |
| `setup_tunnel` が `TunnelException` で失敗 | Gradio 共有サーバ側の障害。数分待って再実行 |
| 動画再生は始まるがシーク/サムネが出ない | `/sprites/<id>.jpg` が 200 を返すか公開 URL 経由で curl で確認 |
| アップロードや変換で 413 / タイムアウト | Drive I/O 負荷。`MAX_CONCURRENT_JOBS=1` を維持、大ファイルは分割して処理 |

## ログ確認 (リアルタイム)

変換や probe の進捗は別スレッドで動くため、セル 6 の実行が終わった後は
そのセル出力には追加のログが流れない（Colab 仕様）。
`setup_logging()` は `/tmp/hls-video.log` にもファイル出力しているので、
別セルで `!tail -f` すれば進捗・警告・エラーがリアルタイムに読める。

- 直近 100 行を眺めるだけ: 下の「ログ末尾」セル
- リアルタイムで追う: 下の「ログをストリーム表示」セル（停止はセルの停止ボタン）

In [ ]:
# ログ末尾（一発実行）
!tail -n 200 /tmp/hls-video.log 2>/dev/null || echo '(ログファイルがまだありません。セル 6 を実行したか確認してください)'

In [ ]:
# ログをストリーム表示（止めるにはセルの停止ボタン ⬛）
!touch /tmp/hls-video.log && tail -n 50 -F /tmp/hls-video.log

## 7. コード更新時の再起動

手元で編集 → `./scripts/sync-to-drive.sh` で Drive に同期したあと、
このセクションのセルを 1 つ実行するだけでアプリを最新コードで再起動できる。

処理内容:
1. 既存の uvicorn サーバを停止（`_HLS_UVICORN.should_exit = True`）
2. `frpc` トンネルプロセスを終了（新しい `*.gradio.live` URL を発行するため）
3. Drive から `LOCAL_ROOT` へコード再コピー、`media/` symlink を作り直し
4. `pip install -e .`（`pyproject.toml` 変更時だけ実質何か起きる）
5. `sys.modules` から `app.*` / `hls_video.*` を破棄して再読込
6. FastAPI + Gradio を再起動、新しい `*.gradio.live` URL を表示

※ Drive マウントや FFmpeg 等の環境構築セルは触らない。Python kernel が
   生きていれば 10 秒ほどで再起動できる。

In [ ]:
# 1) 既存サーバ停止
try:
    _HLS_UVICORN.should_exit = True
    _HLS_THREAD.join(timeout=5)
    print('✓ uvicorn stopped')
except NameError:
    print('(前回起動なし)')
except Exception as e:
    print(f'stop error: {e}')

import subprocess, time
# Gradio の共有トンネル (frpc サブプロセス) を止める
subprocess.run(['pkill', '-f', 'frpc'], check=False)
time.sleep(1)

# 2) Drive からコード再コピー + media symlink 作り直し
import os, shutil
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(
    DRIVE_ROOT, LOCAL_ROOT,
    ignore=shutil.ignore_patterns(
        'media', '.git', '.claude',
        '__pycache__', '.pytest_cache', '.venv', '*.egg-info',
        'node_modules',
    ),
)
drive_media = f'{DRIVE_ROOT}/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{drive_media}/{sub}', exist_ok=True)
os.symlink(drive_media, f'{LOCAL_ROOT}/media')
%cd {LOCAL_ROOT}
print('✓ code re-copied')

# 3) 依存関係更新（pyproject 変更時のため）
!pip install -q -e .

# 4) 既にロード済みのモジュールを sys.modules から破棄 → 次の import で最新が読まれる
import sys
purged = [m for m in list(sys.modules) if m == 'app' or m == 'hls_video' or m.startswith(('app.', 'hls_video.'))]
for m in purged:
    sys.modules.pop(m, None)
print(f'✓ purged {len(purged)} modules')

# 5) FastAPI + Gradio を再起動（セル 6 の内容を再実行）
import gradio as gr
from fastapi import FastAPI
from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

import threading, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)

# 6) 新しい Gradio 共有トンネルを発行
import secrets
from gradio.networking import setup_tunnel
share_url = setup_tunnel(
    local_host='localhost', local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None, share_server_tls_certificate=None,
)
print(f'\n🔄 再起動完了')
print(f'🌐 新しい公開 URL: {share_url}')